In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator
from scipy.stats import kruskal

plt.rcParams["font.family"] = "Times New Roman"
plt.rcParams["axes.unicode_minus"] = False   
plt.rcParams['figure.dpi'] = 300   

def load_and_preprocess_data(file_path):
    """Load and preprocess data"""
    df = pd.read_csv(file_path)
    df['date'] = pd.to_datetime(df['date'], errors='coerce')  
    df = df.sort_values('date').reset_index(drop=True)
    currency = file_path.split('_')[0]
    return df, currency

def calculate_correlations(data, indicators, currency, max_lag=14):
    """Calculate correlation coefficients between indicators and exchange rate"""
    all_results = []
    total_samples = len(data)
    
    for indicator in indicators:
        lag_results = pd.DataFrame({'lag(k)': range(0, max_lag + 1)})
        lag_results['Indicator'] = indicator
        lag_results['Currency'] = currency 
        lag_results['Coefficient'] = np.nan
        
        for lag in range(0, max_lag + 1):
            if lag == 0:
                # Contemporaneous correlation coefficient
                ind_values = data[indicator].values
                rate_values = data['rate'].values
            else:
                # Lagged correlation coefficient
                ind_values = data[indicator].iloc[:-lag].values
                rate_values = data['rate'].values[lag:]
            
            # Ensure data is numeric type
            ind_values = pd.to_numeric(ind_values, errors='coerce')
            rate_values = pd.to_numeric(rate_values, errors='coerce')
            
            # Remove NaN values
            valid_mask = ~(np.isnan(ind_values) | np.isnan(rate_values))
            ind_valid = ind_values[valid_mask]
            rate_valid = rate_values[valid_mask]
            
            # Calculate Pearson correlation coefficient
            if len(ind_valid) > 1 and len(rate_valid) > 1:
                correlation = np.corrcoef(ind_valid, rate_valid)[0, 1]
                lag_results.loc[lag_results['lag(k)'] == lag, 'Coefficient'] = round(correlation, 4)
        
        all_results.append(lag_results)
    
    return pd.concat(all_results, ignore_index=True)

def plot_correlation_subplots(combined_results, filename):
    fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharey=True) 
    axes = axes.flatten()  
    
    indicator_styles = {
        'mean': {'color': '#1f77b4', 'marker': 'o', 'linestyle': '-', 'label': 'Mean Score'},
        'pos': {'color': 'red', 'marker': 'o', 'linestyle': '-', 'label': 'Positive'},
        'neu': {'color': 'orange', 'marker': 'o', 'linestyle': '-', 'label': 'Neutral'},
        'neg': {'color': '#2ca02c', 'marker': 'o', 'linestyle': '-', 'label': 'Negative'}
    }
    
    currency_order = ['EURUSD',  'GBPUSD', 'USDJPY','USDCNY']
    for idx, currency in enumerate(currency_order):
        ax = axes[idx]
        currency_data = combined_results[combined_results['Currency'] == currency]
        
        for indicator in indicator_styles.keys():
            ind_data = currency_data[currency_data['Indicator'] == indicator]
            valid_data = ind_data.dropna(subset=['Coefficient'])
            if len(valid_data) > 0:
                style = indicator_styles[indicator]
                ax.plot(
                    valid_data['lag(k)'],
                    valid_data['Coefficient'],
                    marker=style['marker'],
                    linestyle=style['linestyle'],
                    color=style['color'],
                    markersize=3,
                    linewidth=1,
                    label=style['label']
                )
        
        ax.axhline(y=0, color='gray', linestyle='--', alpha=0.3, linewidth=1)
        ax.set_title(currency, fontsize=12 )
        ax.xaxis.set_major_locator(MultipleLocator(2))
        ax.set_xlim(-0.5, 14.5) 
        ax.tick_params(axis='x', which='major', labelsize=9) 
        ax.yaxis.set_major_locator(MultipleLocator(0.2))
        ax.set_ylim(-0.6, 0.8)  
        
        ax.grid(alpha=0.3, linestyle='--')
        
        ax.set_xlabel('Lag Order', fontsize=10)
        if idx in [0, 2]:
            ax.set_ylabel('Pearson Correlation Coefficient', fontsize=10)
        
        ax.legend(loc='best', fontsize=8, framealpha=0.9) 
    
    plt.tight_layout()
    plt.savefig(filename, bbox_inches='tight', dpi=300)
    plt.show()

def perform_kruskal_wallis_test(data, currency):
    """Perform Kruskal-Wallis test for a single currency"""
    print(f"\n" + "=" * 60)
    print(f"Kruskal-Wallis Test Results - {currency}")
    print("=" * 60)
    
    # Ensure data is numeric type
    pos_data = pd.to_numeric(data['pos'], errors='coerce').dropna().values
    neu_data = pd.to_numeric(data['neu'], errors='coerce').dropna().values
    neg_data = pd.to_numeric(data['neg'], errors='coerce').dropna().values
    
    # Perform Kruskal-Wallis test
    statistic, p_value = kruskal(pos_data, neu_data, neg_data)
    
    print(f"Test Statistic: {statistic:.4f}")
    print(f"P-value: {p_value:.6f}")
    
    # Determine significance
    alpha = 0.05
    if p_value < alpha:
        print(f"Result: At α = {alpha}, reject the null hypothesis.")
        print("The distributions of the three sentiment indicators are significantly different.")
    else:
        print(f"Result: At α = {alpha}, cannot reject the null hypothesis.")
        print("The distributions of the three sentiment indicators are not significantly different.")

def main():
    data_files = ['EURUSD_final.csv', 'GBPUSD_final.csv', 'USDJPY_final.csv','USDCNY_final.csv']
    indicators = ['mean', 'pos', 'neu', 'neg']
    max_lag = 14  # Lag 0-14 periods
    
    print("=" * 80)
    print("Correlation Analysis of Sentiment Indicators with Exchange Rate (Lag 0-14)")
    print("=" * 80)
    
    all_combined_results = []
    
    for file in data_files:
        df, currency = load_and_preprocess_data(file)
        
        # Ensure numeric columns are numeric type
        numeric_columns = ['mean', 'pos', 'neu', 'neg', 'rate']
        for col in numeric_columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
        
        # Extract required columns and clean data
        data = df[['date', 'mean', 'pos', 'neu', 'neg', 'rate']].copy().dropna()
        
        print(f"\n[{currency}]")
        print(f"Sample Size: {len(data)} trading days")
        print(f"Analysis Period: {data['date'].min().strftime('%Y-%m-%d')} to {data['date'].max().strftime('%Y-%m-%d')}")
        
        combined_results = calculate_correlations(data, indicators, currency, max_lag)
        all_combined_results.append(combined_results)
        
        print(f"\n{currency} Correlation Coefficients:")
        pivot_table = combined_results.pivot(index='lag(k)', columns='Indicator', values='Coefficient').round(4)
        print(pivot_table)
        
        # Kruskal-Wallis
        perform_kruskal_wallis_test(data, currency)
        
        print(f"\n{currency} Descriptive Statistics:")
        desc_stats = data[['mean', 'pos', 'neu', 'neg', 'rate']].describe()
        print(desc_stats.round(4))
    
    all_combined_results = pd.concat(all_combined_results, ignore_index=True)
    plot_correlation_subplots(all_combined_results, 'Currencies_Correlation.png')

if __name__ == "__main__":
    main()